# Importing Libraries

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Set display options to show all columns
pd.set_option('display.max_columns', None)

# Loading Data

In [3]:
# Load data
yougov = pd.read_csv("YouGov Data.csv")
oxcgrt = pd.read_csv("OxCGRT Data.csv")
owid = pd.read_csv("OWID Data.csv")

# Data Preparation

In [4]:
# Convert date column to datetime format
yougov["date"] = pd.to_datetime(yougov["date"])
oxcgrt["date"] = pd.to_datetime(oxcgrt["date"])
owid["date"] = pd.to_datetime(owid["date"])

In [ ]:
# Select common variables, common predictors, and outcome-specific variables

# Common demographic and temporal variables
common_vars = [
    "date",
    "age",
    "gender",
    "household_size",
    "employment_status"
]

# Common predictor variables used for all four outcome variables
predictor_vars = [

    # Behavioural & Perception variables
    "covid_dangerous_for_me",
    "likely_to_get_covid",
    "trust_in_government_handling",
    "important_to_improve_health",
    "life_affected_by_covid",

    # Policy variables
    "mask_policy",
    "gathering_restriction",
    "cancel_public_events",
    "stay_home_requirement",
    "public_information_campaign",
    "testing_policy",
    "contact_tracing",
    "stringency_index",

    # Epidemiological variables
    "new_cases_smoothed",
    "new_deaths_smoothed",
    "reproduction_rate",
    "hospital_patients",
    "people_fully_vaccinated_per_hundred"
]

# Outcome-specific datasets
mask_vars = common_vars + predictor_vars + [
    "mask_wearing"
]

handwashing_vars = common_vars + predictor_vars + [
    "wash_hands"
]

crowded_areas_vars = common_vars + predictor_vars + [
    "avoid_crowded_areas"
]

self_isolation_vars = common_vars + predictor_vars + [
    "willingness_self_isolate"
]

In [ ]:
# Merge policy and epidemiological datasets first to preserve daily-level time-series information
# This daily time_df is later used to create correct calendar-day lag variables.
time_df = oxcgrt.merge(owid, on="date", how="outer")
time_df = time_df.sort_values("date").reset_index(drop=True)

# Merge daily policy/epidemiological variables into the individual-level YouGov survey data
merged_df = yougov.merge(time_df, on="date", how="left")

print("Daily time-series dataset shape:", time_df.shape)
print("Merged dataset shape:", merged_df.shape)

Daily time-series dataset shape: (1678, 20)
Merged dataset shape: (53833, 42)


In [ ]:
# Create date features
# These features describe broad pandemic timing without using the inconsistent original survey_week variable.
def create_date_features(df):
    df = df.copy()

    # Broad temporal features
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month

    # Number of days since the first survey date in the dataset
    df["days_since_start"] = (df["date"] - df["date"].min()).dt.days

    # Continuous week index across the full study period
    # This is better than survey_week or week_of_year because it does not reset each year.
    df["week_index"] = (df["days_since_start"] // 7) + 1

    return df


# Apply date feature creation
merged_df = create_date_features(merged_df)

In [ ]:
# Variables for which lag features will be created
lag_columns = [

    # Epidemiological variables
    "new_cases_smoothed",
    "new_deaths_smoothed",
    "reproduction_rate",
    "hospital_patients",
    "people_fully_vaccinated_per_hundred",

    # Policy variables
    "stringency_index",
    "mask_policy",
    "stay_home_requirement",
    "gathering_restriction",
    "cancel_public_events"
]

# Lag periods in calendar days
lag_days = [7, 14]


def create_lag_features(survey_df, daily_time_df, lag_columns, lag_days):
    """
    Creates calendar-day lag features for selected policy and epidemiological variables.

    Lags are created from the daily OxCGRT + OWID time-series dataset first,
    then merged back into the individual-level YouGov survey dataset.

    This ensures lag7 means 7 calendar days before the survey date,
    not 7 survey rows or 7 survey dates before.
    """

    survey_df = survey_df.copy()

    # Keep one row per calendar date from the daily policy/epidemiological data
    daily_lag_df = (
        daily_time_df[["date"] + lag_columns]
        .drop_duplicates(subset="date")
        .sort_values("date")
        .set_index("date")
        .asfreq("D")
    )

    # Fill small gaps in daily time-series data before lagging
    daily_lag_df[lag_columns] = daily_lag_df[lag_columns].ffill().bfill()

    # Create lag variables
    for column in lag_columns:
        for lag in lag_days:
            daily_lag_df[f"{column}_lag{lag}"] = daily_lag_df[column].shift(lag)

    # Keep only lag variables and merge back to survey data by date
    lag_feature_columns = [
        column for column in daily_lag_df.columns
        if "_lag" in column
    ]

    daily_lag_df = daily_lag_df.reset_index()[["date"] + lag_feature_columns]

    survey_df = survey_df.merge(
        daily_lag_df,
        on="date",
        how="left"
    )

    return survey_df


# Create lag features
merged_df = create_lag_features(
    merged_df,
    time_df,
    lag_columns,
    lag_days
)

In [ ]:
# Create interaction variables
def create_interaction_features(df):
    df = df.copy()

    # Policy × Cases
    # Captures whether the effect of policy stringency changes depending on outbreak severity.
    df["policy_x_cases"] = (
        df["stringency_index"] *
        df["new_cases_smoothed"]
    )

    # Risk × Trust
    # Captures whether perceived COVID risk and trust in government jointly influence behaviour.
    df["risk_x_trust"] = (
        df["covid_dangerous_for_me"] *
        df["trust_in_government_handling"]
    )

    return df


# Create interaction features
merged_df = create_interaction_features(merged_df)

In [ ]:
# Add engineered variables to each model dataset
date_features = [
    "year",
    "month",
    "days_since_start",
    "week_index"
]

lag_features = [
    col for col in merged_df.columns
    if "_lag" in col
]

interaction_features = [
    "policy_x_cases",
    "risk_x_trust"
]

engineered_features = (
    date_features +
    lag_features +
    interaction_features
)

In [11]:
# Check if any selected columns are missing before creating final datasets

required_columns = (
    mask_vars +
    handwashing_vars +
    crowded_areas_vars +
    self_isolation_vars +
    engineered_features
)

missing_columns = [
    col for col in required_columns
    if col not in merged_df.columns
]

print("Missing columns:")
print(missing_columns)

Missing columns:
[]


In [12]:
# Create final dataset for each outcome
mask_df = merged_df[mask_vars + engineered_features].copy()
handwashing_df = merged_df[handwashing_vars + engineered_features].copy()
crowded_areas_df = merged_df[crowded_areas_vars + engineered_features].copy()
self_isolation_df = merged_df[self_isolation_vars + engineered_features].copy()

In [13]:
# Remove rows where outcome variable is missing
mask_df = mask_df.dropna(subset=["mask_wearing"])
handwashing_df = handwashing_df.dropna(subset=["wash_hands"])
crowded_areas_df = crowded_areas_df.dropna(subset=["avoid_crowded_areas"])
self_isolation_df = self_isolation_df.dropna(subset=["willingness_self_isolate"])

In [14]:
# Final dataset shape
print("Mask Wearing dataset:", mask_df.shape)
print("Hand Washing dataset:", handwashing_df.shape)
print("Avoid Crowded Areas dataset:", crowded_areas_df.shape)
print("Self Isolation dataset:", self_isolation_df.shape)

Mask Wearing dataset: (53833, 50)
Hand Washing dataset: (52827, 50)
Avoid Crowded Areas dataset: (53833, 50)
Self Isolation dataset: (51765, 50)


# Exporting Data

In [15]:
# Saving the merged data as csv
mask_df.to_csv("Mask Wearing.csv", index=False)
handwashing_df.to_csv("Hand Washing.csv", index=False)
crowded_areas_df.to_csv("Avoid Crowded Areas.csv", index=False)
self_isolation_df.to_csv("Self Isolation.csv", index=False)